# 🏎️ F1 Grand Prix Winner Prediction

Step-by-step notebook walkthrough of the full ML pipeline.

Run each cell in order with **Shift+Enter**.

In [ ]:
# Make sure the src/ module is importable
import sys, os
sys.path.insert(0, os.path.abspath('..'))
print('✅ Ready')

## Step 1 — Fetch Race Data

Downloads results from the Jolpica F1 API (successor to the now-defunct Ergast API). Uses a 0.3s delay between requests to stay within the 4 req/sec rate limit.

In [ ]:
from src.pipeline import fetch_race_results

# Fetch just 3 seasons first to test quickly; change to 2010, 2024 for the full dataset
df_raw = fetch_race_results(season_start=2020, season_end=2024)
print(df_raw.shape)
df_raw.head()

## Step 2 — Explore the Raw Data

In [ ]:
# Who won the most races in our dataset?
df_raw[df_raw['position'] == 1]['driver'].value_counts().head(10)

In [ ]:
# Grid position vs win rate
win_by_grid = df_raw[df_raw['position'] == 1]['grid'].value_counts(normalize=True).sort_index()
win_by_grid.head(5).plot(kind='bar', title='Win rate by starting grid position', color='tomato')

## Step 3 — Feature Engineering

In [ ]:
from src.pipeline import build_features

df = build_features(df_raw.copy())
# Inspect the new columns
df[['driver', 'circuit', 'grid', 'won', 'driver_avg_pos_5', 'circuit_win_rate']].head(15)

## Step 4 — Encode & Split

In [ ]:
from src.pipeline import encode_features, split_data

df, encoders = encode_features(df)
X_train, y_train, X_test, y_test = split_data(df, test_start_year=2023)
print('Features used:', X_train.columns.tolist())

## Step 5 — Train XGBoost Model

In [ ]:
from src.pipeline import train_model

model = train_model(X_train, y_train, X_test, y_test)

## Step 6 — Evaluate

In [ ]:
from src.pipeline import evaluate_model

metrics = evaluate_model(model, X_test, y_test)
print(f"\nROC-AUC: {metrics['roc_auc']:.4f}")

## Step 7 — Predict a Race

In [ ]:
from src.pipeline import predict_next_race

grid = [
    {'driver': 'max_verstappen', 'grid': 1, 'constructor': 'red_bull'},
    {'driver': 'leclerc',        'grid': 2, 'constructor': 'ferrari'},
    {'driver': 'hamilton',       'grid': 3, 'constructor': 'mercedes'},
    {'driver': 'norris',         'grid': 4, 'constructor': 'mclaren'},
    {'driver': 'alonso',         'grid': 5, 'constructor': 'alpine'},
]

predictions = predict_next_race(model, encoders, 'monaco', grid, df, round_num=8)
predictions

## Step 8 — Feature Importance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.pipeline import FEATURES

importances = pd.Series(model.feature_importances_, index=FEATURES)
importances.sort_values().plot(kind='barh', figsize=(8, 5), color='tomato')
plt.title('Feature Importance — F1 Winner Prediction')
plt.tight_layout()
plt.show()